# 06 · Measure real accuracy against independent references

**Goal:** compare frozen decisions on the same visible landmarks and
recording groups, with practical gain and mechanism evidence separated.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](README.md)
· [HAIC setup and launch commands](../../slurm/synthetic-training/README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

## 1. Choose the evaluation split explicitly

`ST_EVALUATION_SPLIT=early` is the initial real decision. Set
`ST_EVALUATION_SPLIT=confirmation` only for the reserved final result.
The source pipeline never calls this notebook. Reference annotations
must be completed independently and selected with `ST_GAVD_ANNOTATIONS`.

Under the strict protocol, early results only determine continue/stop.
Changes chosen using early outcomes turn that portion into labeled
development. Report that distinction in any paper.

In [ ]:
EVALUATION_SPLIT = os.environ.get("ST_EVALUATION_SPLIT", "early")
if EVALUATION_SPLIT not in {"early", "confirmation"}:
    raise ValueError("Choose early or confirmation.")
print(f"Opening independent {EVALUATION_SPLIT} references.")
print(f"Annotation CSV: {cfg.gavd_annotations_csv}")

## 2. Score the same anatomical observations

For each visible landmark, measure the Euclidean pixel distance between
the prediction and the human reference. Divide by the independently
annotated person's box diagonal. Average landmarks within each frame,
then frames within each recording, and finally recordings. Lower is
better. The reference mask and scale are identical across all methods.

Missing predictions receive the configured failure penalty; dropping
their frames would reward model failures. Hidden reference joints are
outside this visible-landmark task. Report annotation coverage and
failed predictions alongside the error table.

Paired recording intervals compare methods on the same recordings.
Many recordings may share one lesson choice, so an interval over
recordings alone does not establish transfer to many new environments.

In [ ]:
evaluation = workflow.evaluate(cfg, split=EVALUATION_SPLIT)
show_result(evaluation)

## 3. Read the evidence in the right order

1. **Practical value:** does the complete method improve on the original
   student and full-budget replay?
2. **Value of selection:** does it improve on fixed, random, or balanced
   synthetic lessons?
3. **Proposed mechanism:** does the full teacher beat the matched selector
   without target prediction change, and the strongest snapshot control?
4. **Transfer:** is that advantage present for the architecture excluded
   from source fitting and validation?
5. **JEPA role:** do frozen video features add value beyond simple context
   and alternative features while response information stays fixed?

A favorable first comparison alone is an adaptation result. Source
ranking differences alone are an opportunity, not proof of successful
real selection. Report effect sizes, uncertainty, and cases where replay
was selected, without converting them into a paper-acceptance probability.